# 02 — Fit Surrogates

Train surrogate models on sweep data from each partial model.

**Prerequisites**: Run `bayesmm run` on all 4 model specs first (see notebook 01).

> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: turn a parameter sweep into a fast probabilistic surrogate per model.
- **Secondary scientific**: explain why the metamodel needs surrogates at all.

## Why surrogates

Joint inference has to evaluate each model thousands of times. The KS model alone takes
seconds per evaluation, so sampling it directly inside MCMC is hopeless.

A surrogate is a cheap probabilistic stand-in fit to a sweep: it predicts the model's
output at unseen inputs *and reports its own uncertainty*. That second part is what
makes it usable in a Bayesian metamodel — the joint posterior needs to know how much to
trust each surrogate, and a point-estimate emulator cannot say.

**Requires a backend**: `pymc` for `pymc_gp`, `sbi` for `sbi_npe`.

In [ ]:
import json
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Run parameter sweeps — at teaching scale

Every surrogate needs training data, and training data means running the real
models across a design. That is the expensive step, and it is worth being honest
about the cost: the **production** design in `specs/model.kinetic_segregation.json`
is 56 points of a C++/Metal simulation, and sweeping all four production specs
takes **over an hour**.

That is the right size for a result you intend to publish, and the wrong size for
a notebook you are reading. So this cell sweeps *reduced copies* of the same
specs — same models, same adapters, same storage, fewer design points — written
to `tmp/tutorial_specs/`. The production specs on disk are untouched.

**What you lose:** a surrogate fitted to a handful of points is a demonstration,
not a result. You will see that directly in its predictive width, which is the
honest signal a GP gives you about how little it knows. When you want the real
thing, sweep `specs/model.*.json` unmodified and refit — nothing else changes.

In [ ]:
# Teaching-scale sweep: reduced copies of the production specs.
#
# Keep at most MAX_LEVELS values per design variable. The KS model dominates the
# cost (56 points of a compiled simulation), so this is what turns an hour into
# roughly a minute while exercising exactly the same code path.
MAX_LEVELS = 2

TUTORIAL_SPECS = ROOT / "tmp" / "tutorial_specs"
TUTORIAL_SPECS.mkdir(parents=True, exist_ok=True)

reduced = []
for spec_path in sorted(SPECS.glob("model.*.json")):
    payload = json.loads(spec_path.read_text())
    grid = payload.get("design", {}).get("grid", {})
    full = 1
    for values in grid.values():
        full *= len(values)
    for var, values in grid.items():
        if len(values) > MAX_LEVELS:
            # The CHEAPEST values, not the endpoints. KS cost scales with
            # `time_sec`: measured on this spec, time_sec=5 takes ~52 s and
            # time_sec=100 takes over 9 minutes, so spanning the range would put
            # the single most expensive simulation in the design in a notebook
            # meant to be read. This biases the training data toward the cheap
            # corner — a real limitation of the teaching sweep, and one more
            # reason the surrogate below is a demonstration, not a result.
            grid[var] = sorted(values)[:MAX_LEVELS]
    small = 1
    for values in grid.values():
        small *= len(values)
    out = TUTORIAL_SPECS / spec_path.name
    out.write_text(json.dumps(payload, indent=2, sort_keys=True))
    reduced.append((spec_path.name, full, small, out))
    print(f"  {spec_path.name:<42} {full:>3} -> {small:>2} points")

print()
for name, _full, _small, out in reduced:
    print(f"Running sweep: {name}")
    r = subprocess.run(
        [sys.executable, "-m", "bayesian_metamodeling.cli.main", "run", str(out)],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f"  {r.stdout.strip().splitlines()[-1] if r.stdout.strip() else 'done'}")
    else:
        print(f"  FAILED: {r.stderr.strip()[:300]}")

## Fit surrogates

In [ ]:
for spec in sorted(SPECS.glob("surrogate.*.json")):
    print(f"Fitting surrogate: {spec.name}")
    r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "fit", str(spec)], cwd=str(ROOT),
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Done")
    else:
        print(f"  FAILED: {r.stderr.strip()[:200]}")

## List trained surrogates

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Publish the surrogates under stable names

`surrogate fit` writes into a **content-addressed** store: each artifact is filed
under a hash of its spec, data and seed, and the registry is keyed by that hash.
That is the right design for provenance — refit with a different seed and you get
a different id, so an artifact can never silently change underneath you.

It is the wrong thing to *reference from a spec you commit to git*, because the
hash changes every time you refit.

So the last step of fitting is to publish each surrogate under a stable name that
`specs/metamodel.tcr_signaling.json` can point at:

```
tmp/surrogate_artifacts/<hash>/artifact.json   ->   artifacts/surrogate_<model>.artifact.json
```

The published copy still carries its `artifact_id`, so provenance survives the
rename — you can always trace a published surrogate back to the exact fit that
produced it. **Notebook 03 cannot run until this cell has.**

In [ ]:
# Publish each freshly-fitted surrogate under the stable name the metamodel spec
# expects. The store is keyed by content hash; the metamodel spec needs a name
# that survives a refit.
import shutil

ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

store = ROOT / "tmp" / "surrogate_artifacts"
by_spec_name = {}
for art in store.glob("*/artifact.json"):
    payload = json.loads(art.read_text())
    spec_name = payload.get("spec_name")
    if not spec_name:
        continue
    # Keep the most recent fit for each surrogate spec.
    prev = by_spec_name.get(spec_name)
    if prev is None or art.stat().st_mtime > prev.stat().st_mtime:
        by_spec_name[spec_name] = art

PUBLISHED = {}
for spec_path in sorted(SPECS.glob("surrogate.*.json")):
    name = json.loads(spec_path.read_text())["name"]
    src = by_spec_name.get(name)
    if src is None:
        print(f"  {name}: NO ARTIFACT — its fit above must have failed")
        continue
    dst = ARTIFACTS / f"{name}.artifact.json"
    shutil.copy2(src, dst)
    PUBLISHED[name] = dst
    aid = json.loads(dst.read_text())["artifact_id"]
    print(f"  {name}: published (artifact_id={aid[:12]}...)")

print()
print(f"{len(PUBLISHED)}/4 surrogates published to {ARTIFACTS.relative_to(ROOT)}/")

## Final check

In [ ]:
# Self-check: the four surrogates the metamodel needs exist, and each really is
# a fitted surrogate for the model it claims — not merely a file that is present.
#
# The previous version asserted `ROOT.is_dir()`, which is true in a repo where
# nothing ran at all. That is the failure this curriculum keeps warning about:
# a green light that proves only that a directory exists.
import json as _json

_spec = _json.loads((SPECS / "metamodel.tcr_signaling.json").read_text())
_missing, _checked = [], []
for _ref in _spec["surrogate_refs"]:
    _p = ROOT / _ref
    if not _p.exists():
        _missing.append(_ref)
        continue
    _a = _json.loads(_p.read_text())
    assert _a.get("artifact_id"), f"{_ref} has no artifact_id"
    _io = _a.get("io_signature") or {}
    assert _io.get("inputs") and _io.get("outputs"), f"{_ref} declares no inputs/outputs"
    _checked.append((_a["spec_name"], _io["inputs"], _io["outputs"]))

assert not _missing, (
    f"metamodel spec references surrogates that were not published: {_missing}. "
    "Re-run the publish cell above; notebook 03 cannot build without them."
)
for _name, _in, _out in _checked:
    print(f"  {_name}: {_in} -> {_out}")
print(f"\n[NB02 self-check OK] {len(_checked)}/4 surrogates fitted and published")